# Part 3.2 — Interpret the Gap

This notebook interprets the longest inactivity gap for each of the ten assigned projects.

For each project, the analysis:
- investigates possible reasons for inactivity and recovery;
- uses commit messages as the primary evidence;
- supplements commit evidence with GitHub issues, pull requests, and README information when useful;
- determines whether the project was Active or Inactive as of September 30, 2025;
- identifies the two dominant themes among the ten most recent commits before that cutoff;
- creates a project-specific reflection describing the observed activity pattern.

In [12]:
import pandas as pd
import numpy as np
from pathlib import Path

NETID = "jmurph91"

PROJECT_SUMMARY_FILE = f"{NETID}_project_summary.csv"
PART3_1_STATS_FILE = "part3_1_outputs/jmurph91_project_stats.csv"

OUTPUT_DIR = Path("part3_2_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# Assignment cutoff
CUTOFF_DATE = pd.Timestamp("2025-09-30 23:59:59")

# Six-month threshold from the assignment
ACTIVE_CUTOFF = pd.Timestamp("2025-04-01")

print(f"Project summary: {PROJECT_SUMMARY_FILE}")
print(f"Part 3.1 statistics: {PART3_1_STATS_FILE}")
print(f"Output folder: {OUTPUT_DIR.resolve()}")
print(f"Assignment cutoff: {CUTOFF_DATE}")

Project summary: jmurph91_project_summary.csv
Part 3.1 statistics: part3_1_outputs/jmurph91_project_stats.csv
Output folder: /home/jmurph91/archeology/MiniProject2/MiniProject2/part3_2_outputs
Assignment cutoff: 2025-09-30 23:59:59


In [13]:
df = pd.read_csv(
    PROJECT_SUMMARY_FILE,
    sep=";"
)

expected_columns = [
    "project_wocid",
    "commit_sha1",
    "author",
    "time",
    "commit message"
]

assert list(df.columns) == expected_columns, (
    f"Unexpected columns.\n"
    f"Expected: {expected_columns}\n"
    f"Found: {list(df.columns)}"
)

df["datetime"] = pd.to_datetime(
    df["time"],
    unit="s",
    errors="coerce"
)

assert df["datetime"].notna().all()

df = df.sort_values(
    ["project_wocid", "datetime"]
).reset_index(drop=True)

print(f"Loaded {len(df):,} commits.")
print(f"Projects: {df['project_wocid'].nunique()}")

Loaded 50,009 commits.
Projects: 10


In [14]:
project_stats = pd.read_csv(
    PART3_1_STATS_FILE,
    sep=";"
)

print("Loaded Part 3.1 statistics:")
print(project_stats.shape)
print()
print(project_stats.columns.tolist())

Loaded Part 3.1 statistics:
(10, 16)

['project', 'number of commits', 'number of authors', 'max time', 'min time', 'number of stars', 'number of forks', 'last commit date', 'LongestGapStart', 'LongestGapEnd', 'LongestGapLength', 'NumCommitsAfterLastGap', 'NumberOfGapsInTimeline', 'ActivityPattern', 'BeforeThemes', 'AfterThemes']


In [15]:
projects = sorted(
    df["project_wocid"].unique()
)

assert set(project_stats["project"]) == set(projects)

print(f"✓ Found all {len(projects)} projects.")
print()
print(projects)

✓ Found all 10 projects.

['angel-ml_angel', 'bmclab_bmc', 'brial_brial', 'comit-network_xmr-btc-swap', 'easystats_insight', 'jasp-stats_jasp-desktop', 'juliarobotics_rigidbodydynamics.jl', 'mmorise_world', 'samreay_chainconsumer', 'systempiper_systempipeshiny']


In [16]:
def classify_theme(message):
    if pd.isna(message):
        return "Other"

    message = str(message).strip().lower()

    bot_words = [
        "dependabot",
        "renovate",
        "github actions",
        "github-actions",
        "bot",
        "automated",
        "auto-update",
        "autoupdate",
        "ci:",
        "ci ",
        "prettier",
        "black formatting",
        "lint"
    ]

    if any(word in message for word in bot_words):
        return "Automated bot contributions"

    security_words = [
        "security",
        "vulnerability",
        "vulnerabilities",
        "cve-",
        "exploit",
        "xss",
        "csrf",
        "authentication vulnerability",
        "security patch"
    ]

    if any(word in message for word in security_words):
        return "Security patches"

    dependency_words = [
        "dependenc",
        "dependency",
        "dependencies",
        "upgrade ",
        "bump ",
        "package",
        "packages",
        "requirements",
        "npm ",
        "pip ",
        "maven",
        "gradle"
    ]

    if any(word in message for word in dependency_words):
        return "Dependency updates"

    release_words = [
        "release",
        "version ",
        "version:",
        "v1.",
        "v2.",
        "v3.",
        "v4.",
        "v5.",
        "publish",
        "published",
        "tag "
    ]

    if any(word in message for word in release_words):
        return "Release"

    documentation_words = [
        "documentation",
        "document",
        "docs",
        "readme",
        "read me",
        "wiki",
        "tutorial",
        "comment",
        "comments",
        "typo"
    ]

    if any(word in message for word in documentation_words):
        return "Documentation updates"

    bug_words = [
        "fix",
        "fixed",
        "fixes",
        "bug",
        "bugs",
        "issue",
        "error",
        "crash",
        "broken",
        "failure",
        "correct",
        "correction",
        "patch"
    ]

    if any(word in message for word in bug_words):
        return "Bug fixes"

    feature_words = [
        "feature",
        "feat",
        "add ",
        "added",
        "implement",
        "implemented",
        "implementation",
        "support",
        "new ",
        "introduce",
        "create",
        "develop"
    ]

    if any(word in message for word in feature_words):
        return "Feature development"

    return "Other"

In [17]:
status_records = []

for project in projects:

    project_df = df[
        (df["project_wocid"] == project) &
        (df["datetime"] <= CUTOFF_DATE)
    ].copy()

    if project_df.empty:
        latest_date = pd.NaT
        status = "Inactive"
    else:
        latest_date = project_df["datetime"].max()

        if latest_date >= ACTIVE_CUTOFF:
            status = "Active"
        else:
            status = "Inactive"

    status_records.append({
        "project": project,
        "LatestCommitAtCutoff": latest_date,
        "Currentstatus": status
    })

status_df = pd.DataFrame(status_records)

print(
    status_df.to_string(index=False)
)

                           project LatestCommitAtCutoff Currentstatus
                    angel-ml_angel  2025-09-29 02:48:20        Active
                        bmclab_bmc  2025-09-15 22:47:25        Active
                       brial_brial  2025-09-29 18:27:13        Active
        comit-network_xmr-btc-swap  2025-08-19 08:50:38        Active
                 easystats_insight  2025-09-30 23:55:44        Active
           jasp-stats_jasp-desktop  2025-09-30 16:12:41        Active
juliarobotics_rigidbodydynamics.jl  2025-03-03 01:38:07      Inactive
                     mmorise_world  2025-02-21 08:30:21      Inactive
             samreay_chainconsumer  2025-09-01 02:54:39        Active
       systempiper_systempipeshiny  2025-05-20 04:19:23        Active


In [18]:
recent_commits = []

for project in projects:

    project_df = df[
        (df["project_wocid"] == project) &
        (df["datetime"] <= CUTOFF_DATE)
    ].copy()

    project_df = project_df.sort_values(
        "datetime"
    )

    recent = project_df.tail(10).copy()

    recent["Theme"] = (
        recent["commit message"]
        .apply(classify_theme)
    )

    recent_commits.append(recent)

recent_commits_df = pd.concat(
    recent_commits,
    ignore_index=True
)

print(
    f"Collected {len(recent_commits_df)} recent commits."
)

Collected 100 recent commits.


In [19]:
def get_recent_themes(project):

    subset = recent_commits_df[
        recent_commits_df["project_wocid"] == project
    ]

    if subset.empty:
        return "N/A"

    counts = (
        subset["Theme"]
        .value_counts()
    )

    top = counts.head(2).index.tolist()

    if len(top) == 1:
        top.append("Other")

    return ":".join(top)


recent_theme_records = []

for project in projects:
    recent_theme_records.append({
        "project": project,
        "RecentThemes": get_recent_themes(project)
    })

recent_theme_df = pd.DataFrame(
    recent_theme_records
)

print(
    recent_theme_df.to_string(index=False)
)

                           project                          RecentThemes
                    angel-ml_angel             Other:Feature development
                        bmclab_bmc           Other:Documentation updates
                       brial_brial                       Other:Bug fixes
        comit-network_xmr-btc-swap                           Other:Other
                 easystats_insight             Feature development:Other
           jasp-stats_jasp-desktop             Other:Feature development
juliarobotics_rigidbodydynamics.jl     Other:Automated bot contributions
                     mmorise_world       Documentation updates:Bug fixes
             samreay_chainconsumer Automated bot contributions:Bug fixes
       systempiper_systempipeshiny          Bug fixes:Dependency updates


In [20]:
for project in projects:

    print("=" * 80)
    print(project)
    print("=" * 80)

    subset = recent_commits_df[
        recent_commits_df["project_wocid"] == project
    ]

    for _, row in subset.iterrows():

        message = str(
            row["commit message"]
        ).split("\n")[0]

        print(
            f"{row['datetime'].date()} | "
            f"{row['Theme']} | "
            f"{message}"
        )

    print()

angel-ml_angel
2025-08-01 | Dependency updates | Fix:correct 'pox.xml' license text and angel graph dependency version
2025-08-04 | Feature development | Merge pull request #1338 from longkezhe/algo#1336
2025-08-18 | Other | Merge c680e55319eed38857e68c264324b3e4a078e831 into 2f2a895654833c95dd1b20bea4928d0c7e566d1b
2025-08-18 | Other | Merge 4cfe0d0d272f21b40a1e60e9bb216b98d42e951b into 2f2a895654833c95dd1b20bea4928d0c7e566d1b
2025-08-18 | Other | Merge dc21377e249582c87a3a93d3e5d4e4400c98d029 into 2f2a895654833c95dd1b20bea4928d0c7e566d1b
2025-08-18 | Other | Merge c455caec07a0e700c39d0abcc21f5406c3747899 into 2f2a895654833c95dd1b20bea4928d0c7e566d1b
2025-09-02 | Feature development | [#1339]Feat: new features of Bruce Force and HNSW implemented by AnnModel
2025-09-03 | Feature development | Merge pull request #1341 from longkezhe/algo#1339
2025-09-25 | Documentation updates | [#1342]Docs: update sona algo document.
2025-09-29 | Documentation updates | Merge pull request #1343 from lo

In [21]:
# ============================================================
# Part 3.2 — Research and interpretation of the longest gap
# ============================================================

# Make sure the project list exists even if earlier cells were
# not executed in the expected order.
projects = sorted(df["project_wocid"].unique())

# Load the Part 3.1 gap-commit file directly.
GAP_COMMITS_FILE = (
    Path("part3_1_outputs")
    / f"{NETID}_project_gap_commits.csv"
)

gap_data = pd.read_csv(
    GAP_COMMITS_FILE,
    sep=";"
)

print(f"Loaded gap commits: {len(gap_data):,}")
print(f"Projects found: {gap_data['project_wocid'].nunique()}")

# ------------------------------------------------------------
# Research findings
# ------------------------------------------------------------

research = {

    "angel-ml_angel": {
        "HypothesizedGapReason":
            "A prolonged development lull after earlier feature and bug-fix work. "
            "The available commit history does not identify a specific issue or event "
            "that caused the 2024-02 through 2025-05 inactivity period.",
        "HasRecovered": "Yes",
        "WhyRecovered":
            "Activity resumed with concentrated bug fixes and new feature development. "
            "The first post-gap commits include fixes to the DeepWalk example and "
            "hash-partition behavior, followed by additional PSF-related features.",
        "WhoRecovered":
            "The selected pre/post samples indicate that recovery involved contributors "
            "different from at least some of the contributors immediately before the gap.",
        "Evidence":
            "Longest gap: 2024-02 through 2025-05 (16 months). "
            "Post-gap commits include DeepWalk fixes, hash-partition fixes, and new "
            "PSF features. Part 3.1 classified post-gap activity primarily as "
            "Feature development and Bug fixes.",
        "Notes":
            "The exact cause of the gap is unclear, but the recovery is clear because "
            "development resumes with both maintenance fixes and new functionality."
    },

    "bmclab_bmc": {
        "HypothesizedGapReason":
            "A temporary development lull. The pre-gap and post-gap commit messages "
            "do not identify a specific issue or event that caused the interruption.",
        "HasRecovered": "Yes",
        "WhyRecovered":
            "The project returned to sustained development after the four-month gap. "
            "Part 2 records substantial activity after the end of the longest gap.",
        "WhoRecovered":
            "The selected pre/post commit evidence does not establish one specific "
            "person as responsible for the recovery.",
        "Evidence":
            "Longest gap: 2016-12 through 2017-03 (4 months). "
            "The project subsequently recorded substantial continued commit activity. "
            "The commit messages do not provide a clear causal explanation for the gap.",
        "Notes":
            "This was one of the harder gaps to interpret because the commit messages "
            "do not provide a clear explanation for the inactivity."
    },

    "brial_brial": {
        "HypothesizedGapReason":
            "A long project pause following an earlier period that included "
            "release-related activity. The available commit messages do not identify "
            "a specific technical failure or external event causing the 27-month gap.",
        "HasRecovered": "Yes",
        "WhyRecovered":
            "Development resumed after the gap, with Feature development becoming "
            "a leading theme in the post-gap sample.",
        "WhoRecovered":
            "The selected pre/post evidence does not establish a single individual "
            "as the driver of the recovery.",
        "Evidence":
            "Longest gap: 2013-02 through 2015-04 (27 months). "
            "Part 3.1 classified pre-gap activity as Other/Release and post-gap "
            "activity as Feature development/Other.",
        "Notes":
            "The unusually long gap makes the exact reason difficult to determine. "
            "The strongest evidence is that the project eventually resumed with "
            "new development."
    },

    "comit-network_xmr-btc-swap": {
        "HypothesizedGapReason": "N/A active Project",
        "HasRecovered": "N/A",
        "WhyRecovered": "N/A",
        "WhoRecovered": "N/A",
        "Evidence":
            "Part 2 found no zero-commit inactivity gap in the complete project timeline.",
        "Notes":
            "There was no longest inactivity gap to investigate."
    },

    "easystats_insight": {
        "HypothesizedGapReason": "N/A active Project",
        "HasRecovered": "N/A",
        "WhyRecovered": "N/A",
        "WhoRecovered": "N/A",
        "Evidence":
            "Part 2 found no zero-commit inactivity gap in the complete project timeline.",
        "Notes":
            "The project shows continuous activity rather than a distinct "
            "inactive/recovery cycle."
    },

    "jasp-stats_jasp-desktop": {
        "HypothesizedGapReason":
            "A very short development pause during the early history of the project. "
            "The one-month gap is too short to support a more specific causal explanation.",
        "HasRecovered": "Yes",
        "WhyRecovered":
            "Development continued immediately after the brief pause and became "
            "highly active over the remainder of the timeline.",
        "WhoRecovered":
            "The project has a large contributor base, so the selected commits do not "
            "support attributing recovery to one individual.",
        "Evidence":
            "Longest gap: 2013-11 through 2013-11 (1 month). "
            "Part 3.1 found Bug fixes among the dominant themes before and after "
            "the gap. The project subsequently accumulated substantial activity.",
        "Notes":
            "This is better interpreted as a brief interruption in an otherwise "
            "highly active development history."
    },

    "juliarobotics_rigidbodydynamics.jl": {
        "HypothesizedGapReason":
            "A prolonged maintenance/development lull. The available commit evidence "
            "does not identify a single event that explains the 2023-07 through "
            "2024-09 gap.",
        "HasRecovered": "Yes",
        "WhyRecovered":
            "Development resumed after the gap, with dependency-related maintenance "
            "remaining a prominent theme.",
        "WhoRecovered":
            "The selected pre/post evidence does not establish a completely new "
            "project team as responsible for the recovery.",
        "Evidence":
            "Longest gap: 2023-07 through 2024-09 (15 months). "
            "Part 3.1 identified Dependency updates as a leading theme in both "
            "the pre-gap and post-gap samples.",
        "Notes":
            "The project resumed after the gap but was classified as inactive by "
            "the 2025-09-30 cutoff."
    },

    "mmorise_world": {
        "HypothesizedGapReason":
            "A long maintenance lull in a relatively small specialized project. "
            "The commit history does not identify a specific issue or event causing "
            "the 15-month gap.",
        "HasRecovered": "Yes",
        "WhyRecovered":
            "The project resumed with maintenance and development activity after "
            "the gap. Post-gap commits included dependency updates and bug fixes.",
        "WhoRecovered":
            "The available evidence does not establish a specific new contributor "
            "as the driver of the recovery.",
        "Evidence":
            "Longest gap: 2022-03 through 2023-05 (15 months). "
            "Part 3.1 classified post-gap activity primarily as Dependency updates "
            "and Bug fixes.",
        "Notes":
            "The project later received maintenance activity, but its latest commit "
            "before the assignment cutoff was February 2025, making it inactive "
            "under the six-month rule."
    },

    "samreay_chainconsumer": {
        "HypothesizedGapReason":
            "A development lull followed by renewed package development. "
            "The available commit evidence does not establish a single event "
            "responsible for the inactivity.",
        "HasRecovered": "Yes",
        "WhyRecovered":
            "Development resumed with bug fixes and package maintenance, followed "
            "by continued project development.",
        "WhoRecovered":
            "The selected evidence does not indicate a complete change in the "
            "project's contributor structure.",
        "Evidence":
            "Longest gap: 2022-10 through 2023-09 (12 months). "
            "Part 3.1 classified Bug fixes as a leading theme in the gap samples. "
            "The project recorded substantial activity after the gap.",
        "Notes":
            "The project clearly recovered after the gap, although its overall "
            "activity pattern is U-shaped rather than consistently increasing."
    },

    "systempiper_systempipeshiny": {
        "HypothesizedGapReason":
            "A temporary development lull associated with the project's release "
            "and maintenance cycle. No specific issue or failure is identified "
            "in the available commit evidence.",
        "HasRecovered": "Yes",
        "WhyRecovered":
            "Development resumed after the five-month gap and continued into 2025.",
        "WhoRecovered":
            "The available evidence does not indicate a complete change in "
            "project ownership or maintainer structure.",
        "Evidence":
            "Longest gap: 2023-11 through 2024-03 (5 months). "
            "Part 3.1 classified pre-gap activity primarily as Dependency updates "
            "and Bug fixes, with continued activity after the gap.",
        "Notes":
            "This appears to be a maintenance/release-cycle interruption rather "
            "than project abandonment."
    }
}


# ------------------------------------------------------------
# Calculate contributor overlap from the actual Part 3.1 data
# ------------------------------------------------------------

for project in projects:

    if project not in research:
        continue

    # No-gap projects do not have pre/post recovery analysis.
    if research[project]["HasRecovered"] == "N/A":
        continue

    subset = gap_data[
        gap_data["project_wocid"] == project
    ]

    pre_authors = set(
        subset.loc[
            subset["pre/post"] == "pre",
            "author"
        ].dropna()
    )

    post_authors = set(
        subset.loc[
            subset["pre/post"] == "post",
            "author"
        ].dropna()
    )

    shared_authors = pre_authors & post_authors

    if len(shared_authors) > 0:
        research[project]["WhoRecovered"] += (
            f" The selected pre/post samples share "
            f"{len(shared_authors)} contributor(s)."
        )
    else:
        research[project]["WhoRecovered"] += (
            " No contributor appears in both selected samples."
        )


# ------------------------------------------------------------
# Build research_df
# ------------------------------------------------------------

research_records = []

for project in projects:

    research_records.append({
        "project": project,
        "HypothesizedGapReason":
            research[project]["HypothesizedGapReason"],
        "HasRecovered":
            research[project]["HasRecovered"],
        "WhyRecovered":
            research[project]["WhyRecovered"],
        "WhoRecovered":
            research[project]["WhoRecovered"],
        "Evidence":
            research[project]["Evidence"],
        "Notes":
            research[project]["Notes"]
    })

research_df = pd.DataFrame(research_records)

print()
print("=" * 80)
print("PART 3.2 RESEARCH SUMMARY")
print("=" * 80)

print(
    research_df[
        [
            "project",
            "HypothesizedGapReason",
            "HasRecovered",
            "WhyRecovered",
            "WhoRecovered"
        ]
    ].to_string(index=False)
)

print()
print("✓ research dictionary created")
print("✓ research_df created")
print("✓ All 10 projects represented")

Loaded gap commits: 160
Projects found: 8

PART 3.2 RESEARCH SUMMARY
                           project                                                                                                                                                                                           HypothesizedGapReason HasRecovered                                                                                                                                                                                                         WhyRecovered                                                                                                                                                                                             WhoRecovered
                    angel-ml_angel        A prolonged development lull after earlier feature and bug-fix work. The available commit history does not identify a specific issue or event that caused the 2024-02 through 2025-05 inactivity period.          Yes Activ

In [22]:
research_df = pd.DataFrame.from_dict(
    research,
    orient="index"
)

research_df.index.name = "project"

research_df = research_df.reset_index()

print(
    research_df.to_string(index=False)
)

                           project                                                                                                                                                                                           HypothesizedGapReason HasRecovered                                                                                                                                                                                                         WhyRecovered                                                                                                                                                                                             WhoRecovered                                                                                                                                                                                                                     Evidence                                                                                                              

# Part 3.2 — Full Analysis and Reflection

## Overall Analysis

This analysis examined the development history of ten open-source projects using World of Code commit data, monthly activity timelines, and the commit activity immediately before and after each project's longest period of inactivity. The analysis treated a gap as a sequence of months with zero commits and identified the longest such gap for each project. It also classified the overall activity pattern of each project as steady, rising, declining, U-shaped, cyclical, or irregular.

The results show that inactivity is common across the projects, but its severity varies considerably. Eight of the ten projects had at least one measurable inactivity gap, while `comit-network_xmr-btc-swap` and `easystats_insight` had no zero-commit gaps in the available timeline. The longest gap overall occurred in `BRiAl/BRiAl`, where activity stopped for 27 months from February 2013 through April 2015. Other substantial gaps occurred in `Angel-ML/angel` (16 months), `JuliaRobotics/RigidBodyDynamics.jl` (15 months), `mmorise/World` (15 months), and `Samreay/ChainConsumer` (12 months).

The gap alone does not necessarily mean that a project was abandoned. Several projects resumed development after long periods of inactivity. For example, BMC recorded 1,372 commits after its longest gap, while JASP Desktop recorded 21,975 commits after its one-month gap. In contrast, World recorded only 11 commits after its longest gap and RigidBodyDynamics.jl recorded 27, suggesting a much weaker recovery. This demonstrates why the number of commits after a gap is useful when interpreting whether a project actually recovered.

The commit themes before and after the gaps also provide evidence about how projects resumed. Angel shifted from a mixture dominated by `Other` and Feature development before the gap to Feature development and Bug fixes afterward. World shifted from Feature development and Other activity to Dependency updates and Bug fixes. BRiAl moved from Other/Release activity toward Feature development/Other activity. SystemPipeShiny remained focused heavily on Dependency updates and Bug fixes before and after its gap. These changes suggest that recovery does not always mean returning to the exact same type of development that existed before the interruption.

The analysis also shows that commit messages alone are often insufficient to determine why a project became inactive. A commit timeline can establish that development stopped, but it usually cannot establish whether the reason was a change in maintainer availability, completion of a development milestone, reduced demand, organizational changes, or another external factor. Therefore, the gap reasons recorded in this analysis are hypotheses based on the available commit evidence rather than confirmed explanations unless the project history directly supports a conclusion.

Contributor activity provides another useful perspective. In some projects, the contributors active immediately after the gap differ from those responsible for the final work before the gap. Angel is a particularly clear example: the pre-gap sample contains substantial work from contributors such as `jyswpp`, while the post-gap sample is dominated by `rachelsun`/`rachelsunrh`. This suggests that contributor changes can be an important part of project recovery. For other projects, the selected commits do not provide enough evidence to attribute recovery to one individual.

Using the assignment's September 30, 2025 cutoff, the projects also differ in their current status. Projects such as `easystats/insight`, `jasp-stats/jasp-desktop`, `BMClab/BMC`, and `BRiAl/BRiAl` had recent activity within the six-month window, while projects such as `mmorise/World` and `JuliaRobotics/RigidBodyDynamics.jl` were classified as inactive because their latest activity before the cutoff was more than six months old. This illustrates an important distinction between historical recovery and current maintenance: a project can recover from a historical gap and still become inactive later.

Overall, the strongest conclusion from the ten projects is that open-source development is often non-linear. Projects can experience long periods of silence and later return to active development, while other projects maintain relatively continuous activity. The combination of monthly activity, gap length, number of gaps, commit themes, contributor changes, and current status provides a more complete picture of project health than commit count alone.

---

## Project-by-Project Analysis

### 1. Angel-ML/angel

`Angel-ML/angel` has 3,418 commits from 147 authors and an overall activity pattern classified as **declining**. Its longest inactivity period lasted 16 months, from February 2024 through May 2025, and there were two inactivity gaps lasting at least three months. The project recorded 54 commits after the longest gap. The pre-gap commits included LCC algorithm development and a DeepWalk bug fix, with the top themes classified as Other and Feature development. After the gap, activity resumed with several concrete fixes and new features, including a DeepWalk example fix, a hash-partition fix, and multiple PSF features. The post-gap themes were Feature development and Bug fixes. The contributor pattern is especially notable because the post-gap work is dominated by `rachelsun`/`rachelsunrh`, whereas the final pre-gap work involved contributors such as `jyswpp`. This suggests that a change in the active contributor group may have helped restart development. The exact reason for the 16-month inactivity period cannot be established from commit messages alone, but the post-gap commits clearly demonstrate a genuine recovery rather than a single isolated maintenance change.

### 2. BMClab/BMC

`BMClab/BMC` contains 2,083 commits from 30 authors and has an **irregular** activity pattern. Its longest gap lasted four months, from December 2016 through March 2017, and the project experienced four gaps of at least three months. Although the selected pre- and post-gap commit messages were mostly project or material names and were therefore classified as Other, the overall timeline shows a strong recovery. There were 1,372 commits after the end of the longest gap, indicating that the four-month interruption did not prevent the project from returning to substantial activity. Because the sampled commit messages provide little information about why the gap occurred, the cause is difficult to interpret with confidence. The most reasonable conclusion is that this was a temporary development lull rather than permanent abandonment. This project demonstrates the limitation of relying only on commit-message themes: the timeline provides strong evidence of recovery even though the messages themselves provide little explanation for the interruption.

### 3. BRiAl/BRiAl

`BRiAl/BRiAl` has 2,999 commits from 34 authors and an **irregular** activity pattern. Its longest inactivity period lasted 27 months, from February 2013 through April 2015, making it the longest gap among all ten projects. The project also had 14 inactivity gaps lasting at least three months, indicating that periods of inactivity are a recurring characteristic of its history. Before the longest gap, the selected commits were dominated by Other and Release activity, while the post-gap sample shifted toward Feature development and Other. The project recorded 186 commits after the gap, showing that development did resume rather than ending permanently. However, the recovery was much smaller than the activity seen in projects such as BMC or JASP Desktop. The available commit evidence does not establish a specific technical or organizational reason for the 27-month pause, so the safest interpretation is that BRiAl experienced a prolonged development lull followed by a later return to feature work. The large number of additional gaps suggests that the project has historically operated in bursts rather than maintaining a continuous development schedule.

### 4. comit-network/xmr-btc-swap

`comit-network/xmr-btc-swap` has 4,980 commits from 68 authors and an overall **U-shaped** activity pattern. Unlike most of the other projects, it has no zero-commit inactivity gaps in the analyzed timeline. Consequently, there is no longest gap to investigate and the assignment's recovery questions are not applicable. The continuous activity is consistent with the project's relatively high historical commit volume and contributor count. Its commit history includes feature work, bug fixes, dependency updates, and automated contributions, showing that development and maintenance occurred continuously rather than stopping for a prolonged period. Because there was no inactivity gap, this project is a useful contrast to the projects with long interruptions. Its U-shaped pattern indicates that the amount of activity changed substantially over time, but those changes did not produce a complete month with zero commits.

### 5. easystats/insight

`easystats/insight` has 9,905 commits from 86 authors and was classified as **steady**. It also has no zero-commit inactivity gaps, making it the second project with no longest inactivity gap to investigate. The lack of zero-commit months suggests a particularly continuous development history compared with the other projects. Because the project remained active throughout the timeline, there is no meaningful recovery event or contributor transition associated with a longest gap. Its steady activity pattern is consistent with a project that receives ongoing development and maintenance rather than relying on occasional bursts of activity. This project demonstrates that a high-volume project can maintain continuity even while individual commit rates vary over time. For the assignment's inactivity analysis, `insight` therefore serves as an example of a project where the appropriate conclusion is not recovery from inactivity, but sustained activity.

### 6. jasp-stats/jasp-desktop

`jasp-stats/jasp-desktop` is the largest project in the dataset by commit count, with 22,023 commits from 242 authors. Its activity pattern is classified as **steady**. The longest inactivity period was only one month, from November 2013 through November 2013, and there were no gaps lasting at least three months. After that one-month interruption, the project recorded 21,975 commits, demonstrating an exceptionally strong continuation of development. The selected pre- and post-gap commits were both dominated by Other and Bug fixes, suggesting that maintenance work remained an important part of the project's development before and after the interruption. Because the gap was only one month long, it would be misleading to describe it as a major project recovery. Instead, it represents a brief interruption in an otherwise highly active project. The large contributor base and very high commit volume also make this project substantially different from smaller repositories in the dataset.

### 7. JuliaRobotics/RigidBodyDynamics.jl

`JuliaRobotics/RigidBodyDynamics.jl` has 2,984 commits from 23 authors and an overall **cyclical** activity pattern. Its longest gap lasted 15 months, from July 2023 through September 2024, and there were four gaps of at least three months. Only 27 commits occurred after the longest gap, making its recovery relatively limited. Dependency updates were an important theme both before and after the gap, with the post-gap sample dominated by Dependency updates and Other activity. This suggests that the project returned primarily through maintenance rather than a large burst of new feature development. The available commit evidence does not identify a specific cause for the 15-month interruption. Importantly, although the project historically recovered from the gap, it was classified as inactive as of September 30, 2025 because its most recent commit before the assignment cutoff was more than six months old. This demonstrates that historical recovery does not necessarily imply long-term continued maintenance.

### 8. mmorise/World

`mmorise/World` has 348 commits from 46 authors and an overall **declining** activity pattern. Its longest gap lasted 15 months, from March 2022 through May 2023, and the project had nine inactivity gaps lasting at least three months. Only 11 commits occurred after the end of the longest gap, making the recovery relatively weak compared with projects such as BMC and JASP Desktop. Before the gap, Feature development was the dominant theme among the selected commits, while post-gap activity shifted toward Dependency updates and Bug fixes. This change suggests that the project returned more through maintenance than through substantial new feature development. The available commit messages do not identify a specific reason for the 15-month interruption. The combination of a declining overall pattern, numerous long gaps, and limited post-gap activity suggests a project whose development became increasingly intermittent over time. Under the assignment's September 30, 2025 cutoff, the project was also inactive because its latest commit before the cutoff was in February 2025.

### 9. Samreay/ChainConsumer

`Samreay/ChainConsumer` has 634 commits from 18 authors and an overall **U-shaped** activity pattern. Its longest inactivity gap lasted 12 months, from October 2022 through September 2023, and there were seven gaps lasting at least three months. The project recorded 242 commits after the longest gap, which provides strong evidence that development resumed substantially. The selected pre- and post-gap commits were both dominated by Bug fixes, suggesting that maintenance and correction of existing functionality were important throughout the interruption and recovery. The available commit messages do not establish one specific event that caused the year-long pause. However, the strong post-gap activity indicates that the project did not remain dormant. The U-shaped pattern is consistent with a project that experienced a significant decline in activity and later returned to a higher level of development. This makes ChainConsumer a good example of why a long inactivity gap should not automatically be interpreted as abandonment.

### 10. systemPipeR/systemPipeShiny

`systemPipeR/systemPipeShiny` has 635 commits from 20 authors and an overall **declining** activity pattern. Its longest gap lasted five months, from November 2023 through March 2024, and there were six gaps lasting at least three months. The project recorded 16 commits after the longest gap. Dependency updates and Bug fixes were the dominant themes both before and after the gap, indicating that the project's activity remained focused largely on maintenance and package compatibility rather than a major change in development direction. The available commit evidence does not identify a specific issue or event responsible for the five-month interruption. The project nevertheless resumed activity after the gap and had commits through May 2025, so it was considered Active under the assignment's September 30, 2025 cutoff. This project illustrates a relatively modest recovery: activity returned, but at a lower overall level than would be expected from a continuously expanding development project.

---

## Final Reflection

The ten repositories demonstrate several distinct patterns of open-source development. Some projects, such as `easystats/insight` and `jasp-stats/jasp-desktop`, show sustained activity with little or no meaningful inactivity. Others, such as `BRiAl/BRiAl`, `Angel-ML/angel`, and `mmorise/World`, contain long periods of silence that make their development histories much more difficult to interpret. A particularly important observation is that a long gap does not necessarily mean permanent abandonment. BMC, ChainConsumer, and Angel all show evidence of meaningful recovery after substantial periods of inactivity.

The longest gaps were generally easy to identify quantitatively because the monthly time series provides a clear boundary between the last active month and the next active month. Determining *why* those gaps occurred was much harder. Commit messages frequently describe what developers changed without explaining why development stopped beforehand. This means that the analysis can confidently report the timing and length of inactivity, but causal explanations should be treated as hypotheses unless supported by additional GitHub issues, pull requests, releases, or project documentation.

Another important finding is that recovery can take different forms. Some projects return with feature development, while others return primarily with dependency updates or bug fixes. Contributor changes can also matter: Angel provides the clearest example of a different contributor group driving post-gap work. Finally, historical recovery should be separated from current project status. A repository can recover from an earlier inactive period and later become inactive again, as seen with RigidBodyDynamics.jl and World under the assignment's September 30, 2025 status rule.

Overall, the project histories support the idea that open-source development is highly variable and often occurs in bursts. Monthly commit activity, inactivity gaps, commit themes, contributor participation, and recent activity each capture a different aspect of project health. Considering all of these measures together gives a more useful picture of project evolution than simply counting the total number of commits.
